In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from gpt4all import GPT4All
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

## 1. Load PDF

In [130]:
loader = PyPDFLoader("4K VF Bullet Camera.pdf")
documents = loader.load()

## 2. Text Splitting

In [132]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [133]:
len(docs)

16

## 3. Load Embedding Model

In [134]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [135]:
docs[0].to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'document', 'Document'],
 'kwargs': {'metadata': {'producer': 'Microsoft® Word for Microsoft 365',
   'creator': 'Microsoft® Word for Microsoft 365',
   'creationdate': '2023-08-27T22:27:48+05:30',
   'moddate': '2023-08-27T22:27:48+05:30',
   'source': '4K VF Bullet Camera.pdf',
   'total_pages': 4,
   'page': 0,
   'page_label': '1'},
  'page_content': '• 4K 1/2.8" CMOS image sensor, low luminance, and \nhigh-definition image \n• Quad Streaming  \n• Intelligent Analytics Supported: People Counting, \nVehicle Counting, Human Detection, Electronic Fence \n(Entry, Exits & Loitering Detection), Line Crossing, \nWrong Direction Detection (Retrograde Detection), \nSafety Helmet Detection, Off-site Detection \n• HTTPS, SSL, Digest Authentication, AES 256 \n• Encryption, RTSP Validation \n• Digital Alarm 1 Ch In/1 Ch Out \n• IP67 & IK10 compliant',
  'type': 'Document'}}

## 4. Store Documents in Vectore Store

In [136]:
vectorstore = FAISS.from_documents(docs, embedding_model)

## 5. Save Vectore Store locally

In [137]:
# 'db' is your FAISS object
vectorstore_path = "faiss_store"

vectorstore.save_local(vectorstore_path)
print("Vector store saved!")

Vector store saved!


## 6. Load Vectore Store 

In [138]:
db = FAISS.load_local(
    "faiss_store",
    embedding_model,
    allow_dangerous_deserialization=True  # <--- safety override
)
print("Vector store loaded!")

Vector store loaded!


## 7. Directly search for top 3 similar documents

In [140]:
docs = db.similarity_search(query, k=3)
for doc in docs:
    print(doc.page_content)
    print('----------------------------------------------')

Exposure Mode Off, High Light correction, Backlight Correction, Range (1-10) 
BLC Mode 1-10 configurable Levels 
White Balance Auto, Manual, Warm light, Natural light, Incandescent lamp, Fluorescent lamp, Locked WB  
Gain Control  Low, MID-Low, MID, MID- HIGH, High 
Sharpness 0-255 
Digital Zoom 25X 
Anti - Flicker Off/ON 
AE Algorithms Shutter Priority, Gain Priority 
Mirror Off, Horizontal Mirror, Vertical Mirror,180-degree flip 
Gamma Mode Curve 1.6, Curve 1.8, Curve 2.0, Curve 2.2
----------------------------------------------
Detection), Line Crossing, Wrong Direction Detection (Retrograde Detection), Safety Helmet Detection, 
Off-site Detection 
Event Actions Enable Alarm Out, Send Email, Start Recording, Play Customized Audio Warning  
PTZ(Manual)  
Pan  0° ~360° 
Tilt 0° ~90° 
Rotation 0° ~360° 
Audio  
Compression G.711U 
Audio Bit Rate 8 – 64 Kbps 
Network  
Network Port  RJ45 10M, 100M Network Self Adjustable
----------------------------------------------
Protocol TCP/IP, IP

## 8. Load llm model

In [141]:
chat = GPT4All("orca-mini-3b-gguf2-q4_0.gguf")

## 9. Test llm model

In [142]:
response = chat.generate("Hello, how are you?")
response

"\nHello, I'm doing well. How about you?"

## 10. Create Prompt 

In [143]:

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""


In [149]:
print(prompt)


Answer the question using the context below.

Context:
Exposure Mode Off, High Light correction, Backlight Correction, Range (1-10) 
BLC Mode 1-10 configurable Levels 
White Balance Auto, Manual, Warm light, Natural light, Incandescent lamp, Fluorescent lamp, Locked WB  
Gain Control  Low, MID-Low, MID, MID- HIGH, High 
Sharpness 0-255 
Digital Zoom 25X 
Anti - Flicker Off/ON 
AE Algorithms Shutter Priority, Gain Priority 
Mirror Off, Horizontal Mirror, Vertical Mirror,180-degree flip 
Gamma Mode Curve 1.6, Curve 1.8, Curve 2.0, Curve 2.2

Detection), Line Crossing, Wrong Direction Detection (Retrograde Detection), Safety Helmet Detection, 
Off-site Detection 
Event Actions Enable Alarm Out, Send Email, Start Recording, Play Customized Audio Warning  
PTZ(Manual)  
Pan  0° ~360° 
Tilt 0° ~90° 
Rotation 0° ~360° 
Audio  
Compression G.711U 
Audio Bit Rate 8 – 64 Kbps 
Network  
Network Port  RJ45 10M, 100M Network Self Adjustable

Protocol TCP/IP, IPv4, IPv6, SSL, TLS, UDP, HTTP/HTTPS,

## 11. Generate Response 

In [146]:

response = chat.generate(prompt)

In [148]:
print(response)


Answer: The PDF contains information about various features of a camera including exposure mode, white balance, sharpness, digital zoom, and gamma mode. It also describes event actions that can be enabled or disabled, such as alarm out, email, start recording, play customized audio warning, pan, tilt, rotation, and audio compression. The PDF mentions protocols for network connections, such as TCP/IP, IPv4, IPv6, SSL, TLS, UDP, HTTP/HTTPS, HTTP-Base64, HTTP-Digest, DHCP, DNS, DDNS, RTP, RTSP, PPPoE, NTP, UPnP, SMTP, FTP, IGMP, Multicast, and system compatibility.
